# Evaluation - sweet.tv TV program recommender

## Evaluation setup

The validation split uses weeks 11-20 as the history window and weeks 21-30 as the target window. The candidate pool is the set of unique tv_show_id values with tv_show_id != 0 in train_schedule_2 (weeks 21-30). This mirrors the test-time constraint where candidates are limited by the test schedule (weeks 31-42).

MAP@5 is computed only for users with a non-empty target set after the candidate-pool filter. AP@5 = sum over hit ranks of (hits so far / rank), divided by min(|target|, 5). Users with an empty target set are excluded from the mean and are not counted in n_users_eval. A single fallback is applied to every method: the global top-5 by history popularity, filtered by the candidate pool. No recommendation is empty.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "data.py").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    RATIO_THRESHOLD,
    build_overlap_pairs,
    items_for_weeks_with_ratio,
    merge_overlapping_views,
)
from src.baselines import (
    ap_at_k,
    build_fallback,
    fill_with_fallback,
    map_at_k,
    recommend_global_pop,
    recommend_personal,
    recommend_recent_pop,
)

RAW_DIR = PROJECT_ROOT / "data_par" / "raw"
EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"
SUBMISSION_PATH = PROJECT_ROOT / "data_par" / "submission.csv"
EVAL_DIR.mkdir(parents=True, exist_ok=True)


def read_existing_json(path):
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as file:
        return json.load(file)


old_eval_metrics = read_existing_json(EVAL_DIR / "eval_metrics.json")
old_sensitivity = read_existing_json(EVAL_DIR / "sensitivity.json")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("EVAL_DIR:", EVAL_DIR)
print("RATIO_THRESHOLD:", RATIO_THRESHOLD)
print("Old eval_metrics loaded:", old_eval_metrics is not None)
print("Old sensitivity loaded:", old_sensitivity is not None)


PROJECT_ROOT: /home/neferpitu/recsys/exam/Quize
RAW_DIR: /home/neferpitu/recsys/exam/Quize/data_par/raw
EVAL_DIR: /home/neferpitu/recsys/exam/Quize/outputs/evaluation
RATIO_THRESHOLD: 0.8
Old eval_metrics loaded: True
Old sensitivity loaded: True


In [2]:
def load_views(path):
    """Load and normalize the view log."""
    frame = pd.read_parquet(path).rename(columns={"duraton": "duration"})
    frame["user_id"] = frame["user_id"].astype("uint64")
    frame["vsetv_id"] = frame["vsetv_id"].astype("int64")
    frame["start_time"] = pd.to_datetime(
        frame["start_time"], errors="coerce"
    ).astype("datetime64[ns]")
    frame["stop_time"] = pd.to_datetime(
        frame["stop_time"], errors="coerce"
    ).astype("datetime64[ns]")
    frame["duration"] = pd.to_numeric(
        frame["duration"], errors="coerce"
    ).fillna(0).astype("int64")
    return frame


def load_schedule(path):
    """Load and normalize a schedule file."""
    frame = pd.read_parquet(path)
    frame["channel_id"] = frame["channel_id"].astype("int64")
    frame["tv_show_id"] = pd.to_numeric(
        frame["tv_show_id"], errors="coerce"
    ).fillna(0).astype("int64")
    frame["start_time"] = pd.to_datetime(
        frame["start_time"], format="%d.%m.%Y %H:%M:%S", errors="coerce"
    ).astype("datetime64[ns]")
    frame["duration"] = pd.to_numeric(
        frame["duration"], errors="coerce"
    ).fillna(0).astype("int64")
    return frame


started = time.time()
views = load_views(RAW_DIR / "dataset11-30week1.parquet")
views_merged = merge_overlapping_views(views)
views_merged["iso_week"] = views_merged["start_time"].dt.isocalendar().week.astype(
    "int16"
)
print("View rows before merge:", int(views.shape[0]))
print("View rows after merge:", int(views_merged.shape[0]))
print(
    "View row reduction percent:",
    round(100.0 * (1.0 - views_merged.shape[0] / views.shape[0]), 4),
)
del views
gc.collect()

train_sched_core = load_schedule(RAW_DIR / "export_arh_11-20-final.parquet")
train_sched_2 = load_schedule(RAW_DIR / "export_arh_21-30-final.parquet")
schedule_11_30 = pd.concat(
    [train_sched_core, train_sched_2], ignore_index=True
).drop_duplicates(subset=["channel_id", "start_time"], keep="first").reset_index(drop=True)
print("train_sched_core rows:", int(train_sched_core.shape[0]))
print("train_sched_2 rows:", int(train_sched_2.shape[0]))
print("schedule_11_30 rows after deduplication:", int(schedule_11_30.shape[0]))

overlap_arrays = build_overlap_pairs(views_merged, schedule_11_30)
pair_view_index = overlap_arrays["view_idx"]
pair_program_index = overlap_arrays["sched_idx"]
pair_overlap = overlap_arrays["overlap_s"]
pair = pd.DataFrame({
    "user_id": views_merged["user_id"].to_numpy()[pair_view_index],
    "tv_show_id": schedule_11_30["tv_show_id"].to_numpy()[pair_program_index],
    "overlap_s": pair_overlap,
    "view_idx": pair_view_index,
    "sched_idx": pair_program_index,
    "view_duration": views_merged["duration"].to_numpy()[pair_view_index],
    "show_duration": schedule_11_30["duration"].to_numpy()[pair_program_index],
})
del overlap_arrays, pair_view_index, pair_program_index, pair_overlap
gc.collect()
print("Positive-overlap pairs:", int(pair.shape[0]))
print("Pair build elapsed seconds:", round(time.time() - started, 2))

View rows before merge: 5821987
View rows after merge: 5725415
View row reduction percent: 1.6587


train_sched_core rows: 1074235
train_sched_2 rows: 1074394
schedule_11_30 rows after deduplication: 2148628


Positive-overlap pairs: 3763634
Pair build elapsed seconds: 100.61


## Candidate pool and ground truth

The candidate pool is taken from train_schedule_2 because the test candidate space will be limited by the test schedule. Without this restriction, baselines can recommend shows that do not air in the target window and MAP is distorted. The ground truth contains aggregated-rule items for weeks 21-30 that belong to the candidate pool; users with no such items are excluded from the metric.

In [3]:
candidate_pool = np.unique(
    train_sched_2.loc[train_sched_2["tv_show_id"] != 0, "tv_show_id"].to_numpy()
)
candidate_set = set(int(value) for value in candidate_pool)
print("Candidate pool size:", len(candidate_set))

val_hist = items_for_weeks_with_ratio(
    pair, views_merged, range(11, 21), RATIO_THRESHOLD
)
val_tgt = items_for_weeks_with_ratio(
    pair, views_merged, range(21, 31), RATIO_THRESHOLD
)
val_tgt = val_tgt[val_tgt["tv_show_id"].isin(candidate_set)].copy()

gt_by_user = {
    int(user): set(int(show) for show in group["tv_show_id"])
    for user, group in val_tgt.groupby("user_id", sort=False)
}
gt_sizes = np.array([len(relevant) for relevant in gt_by_user.values()], dtype=np.int64)
print("History items:", int(val_hist.shape[0]))
print("Target items after candidate filter:", int(val_tgt.shape[0]))
print("Users with non-empty target:", int(len(gt_by_user)))
print("Mean target size:", round(float(gt_sizes.mean()), 4) if len(gt_sizes) else 0.0)

del train_sched_core, train_sched_2, schedule_11_30
gc.collect()

Candidate pool size: 6363


History items: 54294
Target items after candidate filter: 97271
Users with non-empty target: 4504
Mean target size: 21.5966


0

## MAP@5 helper

AP@5 = sum over ranks i where the item is relevant of (hits so far / i), divided by min(|target|, 5). MAP@5 is the mean AP@5 over users with non-empty target sets. Users with an empty target set are skipped and are not part of n_users_eval.

In [4]:
toy_score = ap_at_k([101, 102, 103, 104, 105], {101, 103, 106}, k=5)
toy_expected = (1 / 1 + 2 / 3) / 3
assert np.isclose(toy_score, toy_expected)
print("AP@5 toy check:", round(float(toy_score), 6), "expected:", round(float(toy_expected), 6))


AP@5 toy check: 0.555556 expected: 0.555556


## Unified fallback

The same fallback is used for every method: the global top-5 by history popularity, filtered by the candidate pool. A method-specific or empty fallback would make AP = 0 for personal history an artifact of the method rather than a property of the data. If history popularity has fewer than five candidate items, the fallback is deterministically padded from the candidate pool so every recommendation contains exactly five unique ids.

In [5]:
global_top5 = build_fallback(val_hist, candidate_set, k=5)
print("Global popularity top-5:", global_top5)
print("Fallback length:", len(global_top5))


Global popularity top-5: [200344, 12002955, 10002515, 12002805, 12002856]
Fallback length: 5


## Baselines

All three baselines return exactly five candidate ids and use the same fallback.

- global_pop: top-5 shows by unique users in weeks 11-20, filtered by the candidate pool.
- recent_pop: top-5 shows by unique users in weeks 17-20, filtered by the candidate pool.
- personal: top-5 shows by total_watched_s in weeks 11-20, filtered by the candidate pool, then filled with the fallback.

In [6]:
recent_hist = items_for_weeks_with_ratio(
    pair, views_merged, range(17, 21), RATIO_THRESHOLD
)
recent_top5 = build_fallback(recent_hist, candidate_set, k=5)
global_user_recs = recommend_global_pop(val_hist, candidate_set, k=5)
recent_user_recs = recommend_recent_pop(recent_hist, candidate_set, k=5)

global_recs = {
    int(user): global_user_recs.get(int(user), list(global_top5))
    for user in gt_by_user
}
recent_recs = {
    int(user): recent_user_recs.get(int(user), list(recent_top5))
    for user in gt_by_user
}
personal_recs = recommend_personal(
    val_hist, candidate_set, global_top5, gt_by_user.keys(), k=5
)

for name, recommendation_map in [
    ("global_pop", global_recs),
    ("recent_pop", recent_recs),
    ("personal", personal_recs),
]:
    assert all(len(recommendation_map[user]) == 5 for user in gt_by_user), name

metric_rows = []
for method, recommendation_map in [
    ("global_pop", global_recs),
    ("recent_pop", recent_recs),
    ("personal", personal_recs),
]:
    mean_score, n_users_eval = map_at_k(recommendation_map, gt_by_user, k=5)
    metric_rows.append({
        "method": method,
        "MAP@5": float(mean_score),
        "n_users_eval": int(n_users_eval),
    })

if old_eval_metrics is not None:
    assert len(old_eval_metrics) == len(metric_rows)
    for old_row, new_row in zip(old_eval_metrics, metric_rows):
        assert old_row["method"] == new_row["method"]
        assert abs(old_row["MAP@5"] - new_row["MAP@5"]) <= 1e-12, (old_row, new_row)
        assert old_row["n_users_eval"] == new_row["n_users_eval"]
    print("eval_metrics matches previous values within 1e-12")

eval_metrics = pd.DataFrame(metric_rows)
display(eval_metrics)
with open(EVAL_DIR / "eval_metrics.json", "w", encoding="utf-8") as file:
    json.dump(metric_rows, file, indent=2)
print("Saved:", EVAL_DIR / "eval_metrics.json")
print("Recent popularity top-5:", recent_top5)


eval_metrics matches previous values within 1e-12


,method,MAP@5,n_users_eval
0,global_pop,0.054378,4504
1,recent_pop,0.087168,4504
2,personal,0.043858,4504


Saved: /home/neferpitu/recsys/exam/Quize/outputs/evaluation/eval_metrics.json
Recent popularity top-5: [200344, 240081, 10002515, 6700482, 2400467]


## Sensitivity to target threshold

The 0.8 target threshold is a task hyperparameter, not a rule stated in the competition. It leaves a substantially smaller item set than 0.3 or 0.5. At 0.8 the history contains 54,294 aggregated items and the candidate-filtered target contains 97,271 items; at 0.3 these counts are 239,880 and 250,520. For an ALS or graph model this is a strong reduction of the usable interaction set. The personal baseline is recomputed for thresholds 0.3, 0.5, and 0.8, with the history and fallback rebuilt inside each iteration.

In [7]:
sensitivity_rows = []
for threshold in [0.3, 0.5, 0.8]:
    hist_t = items_for_weeks_with_ratio(
        pair, views_merged, range(11, 21), threshold
    )
    tgt_t = items_for_weeks_with_ratio(
        pair, views_merged, range(21, 31), threshold
    )
    tgt_t = tgt_t[tgt_t["tv_show_id"].isin(candidate_set)].copy()
    gt_t = {
        int(user): set(int(show) for show in group["tv_show_id"])
        for user, group in tgt_t.groupby("user_id", sort=False)
    }
    fallback_t = build_fallback(hist_t, candidate_set, k=5)
    personal_recs_t = recommend_personal(
        hist_t, candidate_set, fallback_t, gt_t.keys(), k=5
    )
    score_t, n_users_eval_t = map_at_k(personal_recs_t, gt_t, k=5)
    mean_gt_size_t = (
        float(np.mean([len(relevant) for relevant in gt_t.values()]))
        if gt_t else float("nan")
    )
    sensitivity_rows.append({
        "threshold": float(threshold),
        "n_hist_items": int(hist_t.shape[0]),
        "n_tgt_items": int(tgt_t.shape[0]),
        "mean_gt_size": mean_gt_size_t,
        "personal_MAP@5": float(score_t),
        "n_users_eval": int(n_users_eval_t),
    })

if old_sensitivity is not None:
    assert len(old_sensitivity) == len(sensitivity_rows)
    for old_row, new_row in zip(old_sensitivity, sensitivity_rows):
        for key in ["threshold", "n_hist_items", "n_tgt_items", "n_users_eval"]:
            assert old_row[key] == new_row[key], (key, old_row, new_row)
        for key in ["mean_gt_size", "personal_MAP@5"]:
            assert abs(old_row[key] - new_row[key]) <= 1e-12, (key, old_row, new_row)
    print("sensitivity matches previous values within 1e-12")

sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
with open(EVAL_DIR / "sensitivity.json", "w", encoding="utf-8") as file:
    json.dump(sensitivity_rows, file, indent=2)
print("Saved:", EVAL_DIR / "sensitivity.json")


sensitivity matches previous values within 1e-12


,threshold,n_hist_items,n_tgt_items,mean_gt_size,personal_MAP@5,n_users_eval
0,0.3,239880,250520,52.630252,0.380815,4760
1,0.5,137234,174566,37.031396,0.223630,4714
2,0.8,54294,97271,21.596581,0.043858,4504


Saved: /home/neferpitu/recsys/exam/Quize/outputs/evaluation/sensitivity.json


## Baseline matrix

The matrix varies the history threshold and the target threshold independently. The candidate pool is fixed to train_schedule_2. The all group contains every user with a non-empty target after the target threshold, and the test group is its intersection with the users in data_par/submission.csv. Recent popularity uses weeks 17-20 inside the history window. The fallback is recomputed for each history threshold.

In [8]:
test_users_frame = pd.read_csv(SUBMISSION_PATH, dtype=str)
test_user_set = set(int(value) for value in test_users_frame["user_id"].astype("uint64"))

hist_cache = {}
recent_cache = {}
target_cache = {}
for hist_threshold in [0.3, 0.5, 0.8]:
    hist_cache[hist_threshold] = items_for_weeks_with_ratio(
        pair, views_merged, range(11, 21), hist_threshold
    )
    recent_cache[hist_threshold] = items_for_weeks_with_ratio(
        pair, views_merged, range(17, 21), hist_threshold
    )
for target_threshold in [0.3, 0.5, 0.8]:
    target_frame = items_for_weeks_with_ratio(
        pair, views_merged, range(21, 31), target_threshold
    )
    target_cache[target_threshold] = target_frame[
        target_frame["tv_show_id"].isin(candidate_set)
    ].copy()

matrix_rows = []
for hist_threshold in [0.3, 0.5, 0.8]:
    hist_items = hist_cache[hist_threshold]
    recent_items = recent_cache[hist_threshold]
    fallback_hist = build_fallback(hist_items, candidate_set, k=5)
    global_top = build_fallback(hist_items, candidate_set, k=5)
    recent_top = build_fallback(recent_items, candidate_set, k=5)
    for target_threshold in [0.3, 0.5, 0.8]:
        target_frame = target_cache[target_threshold]
        gt_all = {
            int(user): set(int(show) for show in group["tv_show_id"])
            for user, group in target_frame.groupby("user_id", sort=False)
        }
        group_truth = {
            "all": gt_all,
            "test": {user: relevant for user, relevant in gt_all.items() if user in test_user_set},
        }
        for group_name, group_gt in group_truth.items():
            users = list(group_gt)
            global_recs = {
                int(user): global_top for user in users
            }
            recent_recs = {
                int(user): recent_top for user in users
            }
            personal_recs = recommend_personal(
                hist_items, candidate_set, fallback_hist, users, k=5
            )
            for method, recommendations in [
                ("global_pop", global_recs),
                ("recent_pop", recent_recs),
                ("personal", personal_recs),
            ]:
                mean_score, n_users_eval = map_at_k(recommendations, group_gt, k=5)
                matrix_rows.append({
                    "method": method,
                    "hist_threshold": float(hist_threshold),
                    "target_threshold": float(target_threshold),
                    "group": group_name,
                    "MAP@5": float(mean_score),
                    "n_users_eval": int(n_users_eval),
                })

eval_matrix = pd.DataFrame(matrix_rows)
assert len(eval_matrix) == 54
with open(EVAL_DIR / "eval_matrix.json", "w", encoding="utf-8") as file:
    json.dump(matrix_rows, file, indent=2)
print("Saved:", EVAL_DIR / "eval_matrix.json")
print("eval_matrix rows:", len(eval_matrix))
print("Users in test group:", len(test_user_set))
for group_name in ["all", "test"]:
    sizes = eval_matrix[eval_matrix["group"] == group_name].groupby("target_threshold")["n_users_eval"].first()
    print(f"n_users_eval by target threshold, group={group_name}:", sizes.to_dict())
for group_name in ["all", "test"]:
    pivot = eval_matrix[
        (eval_matrix["hist_threshold"] == 0.5) & (eval_matrix["group"] == group_name)
    ].pivot(index="method", columns="target_threshold", values="MAP@5")
    print(f"Local MAP@5 pivot, H=0.5, group={group_name}:")
    display(pivot)

del hist_cache, recent_cache, target_cache, test_users_frame
gc.collect()


Saved: /home/neferpitu/recsys/exam/Quize/outputs/evaluation/eval_matrix.json
eval_matrix rows: 54
Users in test group: 1260
n_users_eval by target threshold, group=all: {0.3: 4760, 0.5: 4714, 0.8: 4504}
n_users_eval by target threshold, group=test: {0.3: 1244, 0.5: 1243, 0.8: 1214}
Local MAP@5 pivot, H=0.5, group=all:


target_threshold,0.3,0.5,0.8
method,,,
global_pop,0.29271,0.199652,0.093733
personal,0.30389,0.223630,0.077591
recent_pop,0.36698,0.247813,0.105650


Local MAP@5 pivot, H=0.5, group=test:


target_threshold,0.3,0.5,0.8
method,,,
global_pop,0.339689,0.212328,0.079783
personal,0.346490,0.237493,0.060817
recent_pop,0.421770,0.259895,0.085048


33

## Notes and caveats

- The validation candidate pool is the set of 6,363 unique non-zero tv_show_id values in train_schedule_2.
- The all group contains every user with a non-empty target after the target threshold; the test group is its intersection with the users in data_par/submission.csv.
- For target thresholds 0.3, 0.5, and 0.8, the all group has n_users_eval = 4,760, 4,714, and 4,504; the test group has n_users_eval = 1,244, 1,243, and 1,214. The test group contains 1,260 submission users before the target filter.
- Users with empty target sets are excluded from MAP@5 and are not counted in n_users_eval.
- View rows are merged within each (user_id, vsetv_id) pair before the overlap join; 96,572 rows (1.66%) were removed.
- The 0.8 target threshold is treated as a candidate hyperparameter and is compared with 0.3 and 0.5.
- The history threshold and the target threshold are varied independently in the baseline matrix.
- The evaluation reads no view rows from ISO weeks 31-42 and uses the test schedule only as a reference for candidate availability.